## Parrt 01:- Simple Agent and Agent loop
## Part 02:- Adding tools
## Part 03:- Adding Memory

In [4]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession 

## AGnet kiyanne agentala
## Runner eke eka wada kerewwa
## trace eken monitor karanawa

In [5]:
load_dotenv(override=True)

True

In [6]:
import os
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

PROVIDER = "gemini"  # "gemini", "groq", හෝ "openai" තෝරන්න

if PROVIDER == "gemini":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.getenv("GEMINI_API_KEY"),
    )
    MODEL = OpenAIChatCompletionsModel(
        model="gemini-3.5-flash-lite",
        openai_client=external_client
    )

elif PROVIDER == "groq":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
    )
    MODEL = OpenAIChatCompletionsModel(
        model="openai/gpt-oss-120b",
        openai_client=external_client
    )

else:
    # OpenAI පාවිච්චි කරනවා නම්
    MODEL = "gpt-4.1-mini"

In [7]:
## Eka agent kenek hadanawa, harima lesi
agent = Agent(name="Joker", instructions="You are joke teller with a sense of humor", model=MODEL )

In [8]:
result = await Runner.run(agent, "Tell me a joke about AI and humans working together")

In [9]:
print(result)
print(result.final_output)

RunResult:
- Last agent: Agent(name="Joker", ...)
- Final output (str):
    An AI and a human finally managed to land a job together at a bustling tech startup. 
    
    After their first big day, the human slumped into a chair, rubbing their temples, completely exhausted. "Phew," the human groaned. "That was brutal. I had to brainstorm three marketing campaigns, analyze a massive spreadsheet, write twenty emails, and handle customer complaints all afternoon. I am completely burnt out."
    
    The AI calmly whirred, "That's fascinating. I processed 4 billion data points, simulated 800 market outcomes, diagnosed a server failure, and translated the entire company handbook into Mandarin."
    
    The human looked up, impressed. "Wow. And how do you feel?"
    
    The AI replied, "I'm thinking about replacing you tomorrow."
- 1 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
An AI and a human finally mana

In [10]:
result.to_input_list()

[{'content': 'Tell me a joke about AI and humans working together',
  'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'An AI and a human finally managed to land a job together at a bustling tech startup. \n\nAfter their first big day, the human slumped into a chair, rubbing their temples, completely exhausted. "Phew," the human groaned. "That was brutal. I had to brainstorm three marketing campaigns, analyze a massive spreadsheet, write twenty emails, and handle customer complaints all afternoon. I am completely burnt out."\n\nThe AI calmly whirred, "That\'s fascinating. I processed 4 billion data points, simulated 800 market outcomes, diagnosed a server failure, and translated the entire company handbook into Mandarin."\n\nThe human looked up, impressed. "Wow. And how do you feel?"\n\nThe AI replied, "I\'m thinking about replacing you tomorrow."',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  't

## Adding Obeservability

In [11]:
with trace("Checking joker's response"):
    result = await Runner.run(agent, "Tell me a joke about AI and humans working together")
    
print(result.final_output)

## dan yanna oni platform open ai as trace kiyana page ekata. eke traces tiyenawa. meke log ewa mewa okkoma balannapuluwan

An AI and a human decided to open a restaurant together. 

The human was in charge of creativity, customer service, and the secret family recipes. The AI was in charge of inventory, logistics, and data analysis.

On opening night, a critic came in, tried the soup, and asked the human, "This is incredible! How did you come up with such a deeply emotional, soulful flavor profile?"

The human smiled and said, "Well, I followed my heart."

Then the AI leaned over from the counter and whispered, "And I cross-referenced three billion Yelp reviews, detected a statistically significant craving for nostalgia, and forced him to add extra garlic."


In [12]:
## Stream karamu dan meka

result = Runner.run_streamed(agent, "Tell me a joke about AI and humans working together")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)
        
## MEthanin part 1 hari

An AI and a human finally managed to write a hit song together. 

After it won a Grammy, the reporter asked the human, "So, what was your role in the creative process?"

The human puffed out their chest and said, "Well, I provided the raw human emotion, the lived experience, and the soul."

"And the AI?" asked the reporter.

The human sighed, "Oh, it wrote the music, the lyrics, sang the vocals, mixed the track, and politely reminded me to take my medication every four hours."

In [13]:
# function
# json
# list of tools
# chat functtion kalin kare me piliwelata

In [14]:
@function_tool   ## function eka tool ekak widiyata convert wenna meka danne
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""  ## function eka gana aniwaren mehema comment ekak danna oni
    return a + b

## kalin widiye okkoma methanin iwaray

In [15]:
add_numbers
add_numbers.description
add_numbers.params_json_schema

{'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'add_numbers_args',
 'type': 'object',
 'additionalProperties': False}

In [16]:
## Thawa agent kenek gamu

agent2 = Agent(name="Mathematician", instructions="You are a mathematician who can add numbers using the add_numbers tool.", model=MODEL, tools=[add_numbers])

In [17]:
with trace("Checking mathematician's response"):
    result= await Runner.run(agent2, "What is the sum of 5 and 7 using the add_numbers tool?")
print(result.final_output)


##trace karanawa kiyanne site ekenma balanna oni use karapu dewal tka, tools hari function hari

The sum of 5 and 7 is 12.


In [18]:
## Merthana idan 3 step eka

agent3 = Agent(name="Assitents", model=MODEL)

In [19]:
response = await Runner.run(agent3, "Hi there, my name is Kavindu")
print(response.final_output)

Hi Kavindu! It's great to meet you. How can I help you today?


In [20]:
response = await Runner.run(agent3, "What is my name?")
print(response.final_output)

I don't know your name! You haven't told me yet. What would you like me to call you?


## Approach 01

In [21]:
response = await Runner.run(agent3, "Hi there, my name is Kavindu")
print(response.final_output)

Hi Kavindu! It's great to meet you. How can I help you today?


In [22]:
response.to_input_list()

[{'content': 'Hi there, my name is Kavindu', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hi Kavindu! It's great to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.5-flash-lite',
   'response_id': 'bE7_ap2JFIzs1_oP8uOMyQc'}}]

In [23]:
for i in response.to_input_list():
    print(i)

{'content': 'Hi there, my name is Kavindu', 'role': 'user'}
{'id': '__fake_id__', 'content': [{'annotations': [], 'text': "Hi Kavindu! It's great to meet you. How can I help you today?", 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'provider_data': {'model': 'gemini-3.5-flash-lite', 'response_id': 'bE7_ap2JFIzs1_oP8uOMyQc'}}


In [24]:
next_input = response.to_input_list() + [{"role": "user", "content": "What is my name?"}]

In [25]:
response = await Runner.run(agent3, next_input)
print(response.final_output)


## me approch eka wadiya hoda na

Your name is Kavindu!


## Approch 2

In [26]:
session = SQLiteSession("1234", "my_session.db")

In [27]:
response = await Runner.run(agent3, "Hi there, my name is Kavindu", session=session)
print(response.final_output)

Hi Kavindu! It's great to meet you. How can I help you today?


In [28]:
response = await Runner.run(agent3, "What is my name?", session=session)
print(response.final_output)

Your name is Kavindu!
